In [2]:
import arviz as az
import math
import matplotlib.pyplot as plt
import numpy as np
import pymc as pm
import scipy.stats as stats
import pandas as pd
import rethinking as rt
import xarray as xr
from causalgraphicalmodels import CausalGraphicalModel

RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)
hdi_fill_args = { 'color': 'gray', 'alpha': 0.2 }

__Probabilistic Programming. Wasowski. Pardo. IT University of Copenhagen__

This file contains the list of exercises for the week, as well as any related code.

## Exercises

The exercises for this week are: all exercises from Chapter 7, McElreath 2ed, with exceptions noted below. 

* __7M1__ can be skipped as we did not talk much of AIC.
* __7M2__ is question is poorely formulated, so you can skip it. If interested, this requires reading through Section 7.5 in the textbook. The difference between model selection and model averaging would be more interesting. Unfortunately, model averaging is barely named, and not discussed in depth.  (Kruschke's book shows a simple form of model averaging.)
* __7M3__ One way to demonstrate the phenomenon programmatically is to take one model from previous classes and infer the posterior, then calculate WAIC using `arviz.waic` (https://python.arviz.org/en/stable/api/generated/arviz.waic.html#arviz.waic). After this infer the same model but fitting just to the fragment of the data set (for instance using `head` method of numpy to only get the first 5 data points) and recalculate WAIC again.  `pm.MutableData` and `pm.set_data` can be used to avoid copying the model code twice.
* __7M4__ `arviz.waic` prints the effective number of parameters under `p_waic`
* __7H1__ Just find a linear, quadratic, and cubic regressions and use `arviz.compare` to compare the models (https://python.arviz.org/en/stable/api/generated/arviz.compare.html). The book describes a similar comparison function in Section 7.5, which gives good background (for instance defines 'weight'). Data loading code:

In [3]:
laffer = pd.read_csv("Laffer.csv", delimiter=";")
laffer.head()

,tax_rate,tax_revenue
0,0.07,-0.06
1,8.81,2.45
2,12.84,3.58
3,16.24,2.19
4,19.18,2.46


In [ ]:
# 7M3
# When comparing models with an information criterion, why must all models be fit to exactly
# the same observations? What would happen to the information criterion values, if the models were
# fit to different numbers of observations? Perform some experiments, if you are not sure.

# IF the model is fit with different observations, then the models will not be the same and the
# information criterion values will be different. This is because the information criterion values
# are calculated based on the likelihood of the model given the observations. If the observations
# are different, then the likelihood will be different and so will the information criterion values.

# To demonstrate this, I will fit the same model to different observations and compare the information
# criterion values.

# First, I will fit the model to the first 10 observations.

data = laffer.head(10)

with pm.Model() as m3_1:
    a = pm.Normal("a", mu=0, sigma=100)
    b = pm.Normal("b", mu=0, sigma=10)
    sigma = pm.Uniform("sigma", lower=0, upper=100)
    mu = pm.Deterministic("mu", a + b * data["tax_rate"])
    y = pm.Normal("y", mu=mu, sigma=sigma, observed=data["tax_revenue"])
    trace1 = pm.sample(1000, tune=1000, idata_kwargs={ "log_likelihood": True })
    
# az.summary(trace1, round_to=2, kind="stats")

data_2 = laffer.head(20)

with pm.Model() as m3_2:
    a = pm.Normal("a", mu=0, sigma=100)
    b = pm.Normal("b", mu=0, sigma=10)
    sigma = pm.Uniform("sigma", lower=0, upper=100)
    mu = pm.Deterministic("mu", a + b * data_2["tax_rate"])
    y = pm.Normal("y", mu=mu, sigma=sigma, observed=data_2["tax_revenue"])
    trace1_2 = pm.sample(1000, tune=1000, idata_kwargs={ "log_likelihood": True })

Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [a, b, sigma]


Output()

Sampling 4 chains for 1_000 tune and 1_000 draw iterations (4_000 + 4_000 draws total) took 36 seconds.
There were 6 divergences after tuning. Increase `target_accept` or reparameterize.
Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [a, b, sigma]


Output()

Sampling 4 chains for 1_000 tune and 1_000 draw iterations (4_000 + 4_000 draws total) took 36 seconds.


In [14]:
az.waic(trace1, pointwise=True, scale="deviance")

c:\Users\patr7\miniconda3\envs\prpro-2025\Lib\site-packages\arviz\stats\stats.py:1653: UserWarning: For one or more samples the posterior variance of the log predictive densities exceeds 0.4. This could be indication of WAIC starting to fail. 
See http://arxiv.org/abs/1507.04544 for details
  warnings.warn(


Computed from 4000 posterior samples and 10 observations log-likelihood matrix.

              Estimate       SE
deviance_waic    33.81     3.55
p_waic            2.69        -

There has been a warning during the calculation. Please check the results.

In [ ]:
az.waic(trace1_2, pointwise=True, scale="deviance")

c:\Users\patr7\miniconda3\envs\prpro-2025\Lib\site-packages\arviz\stats\stats.py:1653: UserWarning: For one or more samples the posterior variance of the log predictive densities exceeds 0.4. This could be indication of WAIC starting to fail. 
See http://arxiv.org/abs/1507.04544 for details
  warnings.warn(


Computed from 4000 posterior samples and 20 observations log-likelihood matrix.

              Estimate       SE
deviance_waic    86.50    12.30
p_waic            3.78        -

There has been a warning during the calculation. Please check the results.

In [16]:
# 7M4
# What happens to the effective number of parameters, as measured by PSIS or WAIC, as a prior
# becomes more concentrated? Why? Perform some experiments, if you are not sure.

with pm.Model() as m4:
    a = pm.Normal("a", mu=0, sigma=2)
    b = pm.Normal("b", mu=0, sigma=1)
    sigma = pm.Uniform("sigma", lower=0, upper=20)
    mu = pm.Deterministic("mu", a + b * data["tax_rate"])
    y = pm.Normal("y", mu=mu, sigma=sigma, observed=data["tax_revenue"])
    trace2 = pm.sample(1000, tune=1000, idata_kwargs={ "log_likelihood": True })

Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [a, b, sigma]


Output()

Sampling 4 chains for 1_000 tune and 1_000 draw iterations (4_000 + 4_000 draws total) took 38 seconds.


In [18]:
az.waic(trace1, pointwise=True, scale="deviance")

c:\Users\patr7\miniconda3\envs\prpro-2025\Lib\site-packages\arviz\stats\stats.py:1653: UserWarning: For one or more samples the posterior variance of the log predictive densities exceeds 0.4. This could be indication of WAIC starting to fail. 
See http://arxiv.org/abs/1507.04544 for details
  warnings.warn(


Computed from 4000 posterior samples and 10 observations log-likelihood matrix.

              Estimate       SE
deviance_waic    33.81     3.55
p_waic            2.69        -

There has been a warning during the calculation. Please check the results.

In [17]:
az.waic(trace2, pointwise=True, scale="deviance")

c:\Users\patr7\miniconda3\envs\prpro-2025\Lib\site-packages\arviz\stats\stats.py:1653: UserWarning: For one or more samples the posterior variance of the log predictive densities exceeds 0.4. This could be indication of WAIC starting to fail. 
See http://arxiv.org/abs/1507.04544 for details
  warnings.warn(


Computed from 4000 posterior samples and 10 observations log-likelihood matrix.

              Estimate       SE
deviance_waic    32.65     3.10
p_waic            2.16        -

There has been a warning during the calculation. Please check the results.

In [ ]:
# 7M5
# Provide an informal explanation of why informative priors reduce overfitting

# We use informative priors to constrain the model and remove outliers.

# Informative priors reduce overfitting because they constrain the model to a smaller set of
# plausible parameters. This means that the model is less likely to fit the noise in the data
# and more likely to fit the underlying structure of the data. This reduces the risk of overfitting

# 7M6
# Provide an informal explanation of why overly informative priors result in underfitting.

# If we constrain the model too much, then we will be losing information that is present in the data.
# This will result in underfitting because the model will not be able to capture the underlying
# structure of the data.

I added `standardize` to `rethinking.py` and imported it above as `rt` so you can call `rt.standardize` to standardize data.

* __7H2__ To find the importance of each point you can use PSIS and its k parameter.  To get it with Pymc inger the posterior with log-likelihood (add parameter `idata_kwargs = { 'log_likelihood': True }` to `sample`). this adds the `log_likelihood` group to the inference data. Then use `arviz.psislw` as follows)

lw_out, kss = az.psislw(-idata.log_likelihood.stack(__sample__=["chain", "draw"]).revenue)

`kss.values` contains the k values for each input data point.

* __7H3__ can be solved on paper, but you can also do calculations in a very simple notebook (it seems no inference library is needed). Here are the distributions for different islands for your convenience:

In [ ]:
IB = [[0.2 , 0.2 , 0.2 , 0.2 , 0.2],
      [0.8 , 0.1 , 0.05 , 0.025 , 0.025], 
      [0.05 , 0.15 , 0.7 , 0.05 , 0.05]]
for i in range(3):
    for j in range(5):
        print(f"{IB[i][j]:.2f} ", end = "")
    print()

* __7H4__ 

First the code that generates the data:

In [ ]:
def inv_logit(x):
    return np.exp(x) / (1 + np.exp(x))

In [ ]:
def sim_happiness(N_years = 100, seed = 1234):
    np.random.seed(seed)
   
    # Initial population setup
    # the number of persons in the sample
    N = 20 * 65 
    # 20 persons in each age below 65
    age = np.repeat(np.arange(65), 20)
    # each person in each age has a different happiness level between -2 and +2
    happiness = np.repeat(np.linspace(-2, 2, 20), 65)
    married = np.zeros(N, dtype=bool)

    for i in range(N_years):
        # age population
        age = age + 1
        # replace old folk with new folk
        ind = (age == 65)
        age[ind] = 0
        married[ind] = False
        happiness[ind] = np.linspace(-2, 2, 20)
        
        # do the work
        elligible = (married == 0) & (age >= 18)
        marry = np.random.binomial(1, inv_logit(happiness[elligible] - 4)) == 1
        married[elligible] = marry
        
    popn = xr.Dataset({
        "age": ("index", age),
        "happiness": ("index", happiness),
        "married": ("index", married),
    },
    coords={ "index": np.arange(N) })
    
    # Sort the Dataset by 'age'
    sorted_indices = np.argsort(popn.age.values)
    popn_sorted = popn.isel(index=sorted_indices)    
    
    return popn_sorted

In [ ]:
popn = sim_happiness()
# We add a column translating the married field from Boolean to integer
popn['mid'] = popn.married.astype(int)
# We reschale age, like the book does
popn['A'] = (popn.age - 18) / (65 - 18)
az.summary(popn.to_dataframe().to_dict(orient="list"), kind="stats", round_to=2)

In [ ]:
popn

Models m6.9 and m6.10 as not shown in the lecture:

In [ ]:
with pm.Model() as m_6_9:
    α = pm.Normal("α", 0, 1, shape = 2)
    β = pm.Normal("β", 0, 2)
    σ = pm.Exponential("σ", 1)
    μ = α[popn.mid.values] + β * popn.A.values
    happiness = pm.Normal("happiness", mu = μ, sigma = σ, observed = popn.happiness.values)
    idata_m_6_9 = pm.sample(random_seed = rng, idata_kwargs = { 'log_likelihood': True })

In [ ]:
az.summary(idata_m_6_9)

The model does seem to find clear negative association of age and  happiness.

In [ ]:
with pm.Model() as m_6_10:
    α = pm.Normal("α", 0, 1)
    β = pm.Normal("β", 0, 2)
    σ = pm.Exponential("σ", 1)
    μ = α + β * popn.A.values
    happiness = pm.Normal("happiness", mu = μ, sigma = σ, observed = popn.happiness.values)
    idata_m_6_10 = pm.sample(random_seed = rng, idata_kwargs = { 'log_likelihood': True })

In [ ]:
az.summary(idata_m_6_10)

This model (m_6_10) finds no association between age and happiness

With all this above the exercise is quite simple. Use `az.compare` or `az.waic`

* __7H5__ Code for loading data

In [ ]:
foxy = pd.read_csv("foxes.csv", delimiter=";").to_xarray()

Standardize the data

In [ ]:
foxy['area_std'] = rt.standardize(foxy.area)
foxy['weight_std'] = rt.standardize(foxy.weight)
foxy['avgfood_std'] = rt.standardize(foxy.avgfood)
foxy['groupsize_std'] = rt.standardize(foxy.groupsize)
foxy

In [ ]:
S = 5000

Code for Point 2, 4, and 5. Add models for point 1 and 3 above yourself.

**Point 2 (avgfood + groupsize; Ex 6H5)**

In [ ]:
with pm.Model() as m_2:
    α = pm.Normal("α", 0, 0.2)
    βf = pm.Normal("βf", 0, 0.5)
    βg = pm.Normal("βg", 0, 0.5)
    σ = pm.Exponential("σ", 1.0)

    μ = α + βf * foxy.avgfood_std.values + βg + foxy.groupsize_std.values
    weight_std = pm.Normal ("weight_std", mu = μ, sigma = σ, observed = foxy.weight_std.values)
    idata_2 = pm.sample(S, random_seed = rng, idata_kwargs = { 'log_likelihood': True })

**Point 4 (avgfood; Ex 6H4)**

In [ ]:
with pm.Model() as m_4:
    α = pm.Normal("α", 0, 0.2)
    β = pm.Normal("β", 0, 0.5)
    σ = pm.Exponential("σ", 1.0)

    μ = α + β * foxy.avgfood_std.values
    weight_std = pm.Normal ("weight_std", mu = μ, sigma = σ, observed = foxy.weight_std.values)
    idata_4 = pm.sample(S, random_seed = rng, idata_kwargs = { 'log_likelihood': True })

**Point 5 (area; Ex 6H3)**

In [ ]:
with pm.Model() as m_5:
    α = pm.Normal("α", 0, 0.2)
    β = pm.Normal("β", 0, 0.5)
    σ = pm.Exponential("σ", 1.0)

    μ = α + β * foxy.area_std.values
    weight_std = pm.Normal ("weight_std", mu = μ, sigma = σ, observed = foxy.weight_std.values)
    idata_5 = pm.sample(S, random_seed = rng, idata_kwargs = { 'log_likelihood': True })

Now use az.compare or az.waic